<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Deep Learning & Neural Networks** · Week 2 · Training Deep Networks Well · Lesson 11 of 24

# Datasets, DataLoaders and Tabular Data

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

**This lesson uses a GPU.** Before you start: **Runtime → Change runtime type → T4 GPU → Save**. The code also runs without a GPU, only more slowly.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks).

**Learning objective:** Feed data to PyTorch with `Dataset` and `DataLoader`, prepare tabular data correctly, and split data into training, validation and test sets.

## Why not just slice tensors?

Until now we shuffled indices by hand. Real projects use PyTorch's two data tools:

- **Dataset** — knows how many examples there are and how to get example number i.
- **DataLoader** — wraps a Dataset and gives you shuffled **mini-batches**, optionally loading them in parallel.

Images that do not fit in memory, text files, audio — all are handled the same way: write a Dataset, wrap it in a DataLoader, and the training loop never changes.

## The loans data again

We use the mobile-money micro-loans data from Machine Learning Fundamentals: predict `defaulted` (1 = did not repay).

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 2000
income = rng.lognormal(12.2, 0.45, n).round(-3)
loan = rng.uniform(20_000, 300_000, n).round(-3)
months = rng.integers(1, 61, n)
late = rng.poisson(0.6, n)
salary = rng.random(n) < 0.55
z = -3.2 + 2.5 * (loan / income) - 0.03 * months + 0.8 * late - 0.9 * salary
default = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
loans = pd.DataFrame({"income_rwf": income, "loan_rwf": loan, "months_customer": months,
                      "late_payments": late, "salaried": salary.astype(int),
                      "defaulted": default})
print(loans.shape, "default rate:", round(loans["defaulted"].mean(), 3))
loans.head()

## Prepare tabular data for a network

Networks are as sensitive to scale as k-NN: income in hundreds of thousands next to a 0/1 column would make training slow and unstable. **Standardise**, using the training rows only:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = loans.drop(columns="defaulted").to_numpy()
y = loans["defaulted"].to_numpy()
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.2, random_state=42, stratify=y_dev)

scaler = StandardScaler().fit(X_train)            # learn from training rows only
X_train, X_val, X_test = (scaler.transform(a) for a in (X_train, X_val, X_test))
print(len(X_train), len(X_val), len(X_test))

## A custom Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class LoanDataset(Dataset):
    def __init__(self, features, labels):
        self.x = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x[i], self.y[i]

train_ds = LoanDataset(X_train, y_train)
print(len(train_ds), train_ds[0])

Only two methods are needed: `__len__` and `__getitem__`.

## A DataLoader

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(LoanDataset(X_val, y_val), batch_size=256)

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)          # one batch: 64 rows of 5 features
print("batches per epoch:", len(train_loader))

- `shuffle=True` for training (a new order every epoch); no shuffling for validation and test.
- `num_workers=2` loads batches in parallel — useful for images on disk.
- For data already in tensors, `TensorDataset(x, y)` saves writing a class.

## Train with the loader

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(5, 32), nn.ReLU(), nn.Linear(32, 1))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(30):
    model.train()
    for xb, yb in train_loader:                 # the loop is now very clean
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(xb).squeeze(1), yb)
        loss.backward()
        opt.step()

model.eval()
with torch.no_grad():
    val_logits = torch.cat([model(xb).squeeze(1) for xb, _ in val_loader])
val_acc = ((val_logits > 0).float().numpy() == y_val).mean()
print("validation accuracy:", round(val_acc, 3))

## Categories in tabular networks

For a column such as district, either one-hot encode it (as in Machine Learning Fundamentals) or use an **embedding** layer that learns a small vector for each category — the same idea used for words in Week 4.

## Should you use a network for tables?

Honest advice from industry and research: on ordinary tables of a few thousand rows, **gradient boosting and logistic regression are usually as good or better** than neural networks, and they are faster and easier to explain. Networks shine with images, sound, text and very large or mixed datasets. The next lesson's project tests this claim.

## Key terms

- **Dataset** — gives the number of examples and example i.
- **DataLoader** — turns a Dataset into shuffled mini-batches.
- **TensorDataset** — a ready-made Dataset for tensors.
- **Batch size** — examples per update.
- **Embedding** — a learned vector for each category.

**Takeaway:** Wrap your data in a Dataset and a DataLoader, scale features with training statistics only, and the same clean training loop works for any data.

## Try it yourself

1. Replace `LoanDataset` with `TensorDataset` and check that training gives the same result.
2. Train with batch sizes 8, 64 and 512. How do speed and validation accuracy change?
3. Remove the StandardScaler and train again. What happens?
4. Add the engineered feature `loan_to_income` from Machine Learning Fundamentals before scaling. Does the network improve?

---
Finished? Go back to [Deep Learning & Neural Networks on WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*